## 네이버 금융 종목 뉴스 크롤러

수집 로직은 `preprocessing/crawl.py`에 있고 이 노트북은 그것을 불러 쓰기만 한다.
코드를 두 벌 두면 한쪽만 고치는 사고가 나므로, 셀렉터·재시도·캐시 규칙은 전부
스크립트 한 곳에만 있다.

터미널에서 그냥 돌려도 된다.

```bash
uv run python preprocessing/crawl.py                                    # 전체
uv run python preprocessing/crawl.py --stocks 005930 --pages 2 --per-stock 5   # 맛보기
```

노트북 쪽이 나은 경우는 진행 상황을 보면서 중간에 멈추거나, 수집 직후 결과를
바로 들여다볼 때다.

### 수집 규모

종목 10개 × 목록 25페이지 × 최대 60건. 요청 간격 1초 기준 **20~30분** 걸린다.

중간에 끊겨도 `data/raw/crawl_cache/`에 종목별로 저장돼 있어 다시 실행하면 이어서
진행한다. 처음부터 다시 받으려면 그 폴더를 지운다.

### 준비

의존성은 uv가 관리한다. `!pip install`은 잠긴 버전과 어긋나므로 쓰지 않는다.

```bash
uv sync             # requests, beautifulsoup4, ipykernel
uv run jupyter lab  # 커널은 프로젝트 .venv 선택
```

경로는 `pyproject.toml`이 있는 폴더를 루트로 잡는다. 루트에서 열든
`preprocessing/`에서 열든 같은 `data/`를 쓴다.

In [ ]:
# 노트북에는 __file__이 없으므로 프로젝트 루트를 찾아 import 경로에 넣는다.
import sys

from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from preprocessing import crawl

print(f"종목 {len(crawl.STOCKS)}개 · 목록 {crawl.LIST_PAGES}페이지 · 종목당 {crawl.ARTICLES_PER_STOCK}건")
print(f"산출물: {crawl.OUT_PATH}")
print(f"중간 저장: {crawl.CACHE_DIR}")

In [ ]:
# 전체 수집 (20~30분). 종목마다 진행 상황이 찍히고, 캐시가 있으면 건너뛴다.
# 맛보기로 돌리려면 아래를 쓴다:
#   crawler = crawl.Crawler({"005930": "삼성전자"}, pages=2, per_stock=5)
import json

crawler = crawl.Crawler()
articles = crawler.crawl()

crawl.OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
crawl.OUT_PATH.write_text(json.dumps(articles, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n총 {len(articles)}건 저장 완료 → {crawl.OUT_PATH}")
crawl.summarize(articles)

In [ ]:
# 수집 직후 바로 확인 — 노트북으로 돌릴 때의 이점
import collections

print("종목별 건수")
for name, count in collections.Counter(a["stock_name"] for a in articles).items():
    print(f"  {name:16} {count:>3}건")

lengths = sorted(len(a["content"]) for a in articles)
print(f"\n본문 길이 중앙값 {lengths[len(lengths)//2]:,}자 · 최소 {lengths[0]:,} · 최대 {lengths[-1]:,}")
print(f"중복 URL: {len(articles) - len({a['url'] for a in articles})}건")
print("\n다음: uv run python preprocessing/clean.py")